# Tema: Vistas persistentes y temporales

## Objetivos
Comparar duración, materialización y contexto de resolución.

## Conceptos importantes para el examen
TEMP VIEW vive en sesión; VIEW registra una consulta; materialized view almacena resultados y requiere actualización gestionada.

**Dificultad:** Básico · **Tiempo estimado:** 45 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_11_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Vista temporal

In [ ]:
employees.filter("active").createOrReplaceTempView("active_temp")
display(spark.sql("SELECT COUNT(*) FROM active_temp"))

### 2. Vista persistente

In [ ]:
%sql
CREATE OR REPLACE VIEW active_persistent AS SELECT employee_id, department, salary FROM employees WHERE active;
SELECT * FROM active_persistent;

### 3. Tabla snapshot frente a vista

In [ ]:
%sql
CREATE TABLE active_snapshot USING DELTA AS SELECT * FROM active_persistent;
UPDATE employees SET salary = 99000 WHERE employee_id = 1;
SELECT 'view' object, salary FROM active_persistent WHERE employee_id=1
UNION ALL SELECT 'table', salary FROM active_snapshot WHERE employee_id=1;

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea una vista temporal con empleados de Data.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Crea una vista persistente con salario medio por departamento.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Modifica un salario de Data y compara la vista con una tabla snapshot creada antes.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Muestra la definición de salary_summary y localízala en Catalog Explorer.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Elimina data_temp y recrea otra sesión mentalmente: ¿qué objetos permanecerían? Comprueba que employees sigue existiendo.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** createOrReplaceTempView o CREATE TEMP VIEW.

**Pista 2:** GROUP BY en CREATE VIEW.

**Pista 3:** La tabla no se recalcula por ser CTAS.

**Pista 4:** SHOW CREATE TABLE también muestra definiciones de vistas.

**Pista 5:** DROP VIEW no borra la tabla base.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW data_temp AS SELECT * FROM employees WHERE department='Data';
SELECT * FROM data_temp;

### Solución 2

In [ ]:
%sql
CREATE OR REPLACE VIEW salary_summary AS SELECT department, AVG(salary) avg_salary FROM employees GROUP BY department;
SELECT * FROM salary_summary;

### Solución 3

In [ ]:
%sql
CREATE OR REPLACE TABLE summary_snapshot USING DELTA AS SELECT * FROM salary_summary;
UPDATE employees SET salary=80000 WHERE employee_id=3;
SELECT * FROM salary_summary;
SELECT * FROM summary_snapshot;

### Solución 4

In [ ]:
%sql
SHOW CREATE TABLE salary_summary;
SHOW VIEWS;

### Solución 5

In [ ]:
%sql
DROP VIEW data_temp;
SELECT COUNT(*) FROM employees;
-- Persisten employees y salary_summary; la vista temporal no sobrevive a una sesión nueva.
-- Una materialized view (tema 22) guarda resultados y los refresca mediante su infraestructura.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué objeto desaparece con su sesión?

A. Tabla Delta

B. Vista temporal

C. Vista persistente

D. Volumen

### Pregunta 2
¿Qué se actualiza al consultar tras cambiar la tabla base?

A. CTAS automáticamente

B. Un CSV exportado

C. Vista SQL normal

D. Copia manual

### Pregunta 3
¿Qué caracteriza una materialized view?

A. Persiste resultados de una consulta

B. Siempre es temporal

C. No necesita refresco nunca

D. Solo admite nombres de una parte

### Respuestas y explicación
**1. B** — Su alcance es la sesión que la registra.

**2. C** — La consulta de la vista se evalúa sobre los datos actuales.

**3. A** — Su mantenimiento actualiza los resultados almacenados.

## PARTE 6 - RETO FINAL
Ofrece a BI un resumen mediante vista y una instantánea Delta. Modifica la fuente y documenta las diferencias de actualización y coste de lectura.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
